### Run trajectories from M2 mooring *(08.2018 - 31.12.2021)*

Computing back-trajectories from the M2 mooring every two days from 1 Aug 2018 through December 2021.

- Using a **2km IBCAO coastline excluding Svalbard** to check interaction with coasts (*MASK*)
- Updated scheme - now using KDtree to identify the 4 nearest non-land-masked points and comuuting a radially distance-averaged mean based all sea ice covered points among these.
    - Confirmed that this doesn't qualitatively change anything.    
- End criteria (*ABC*): 
    
    - Nearest non-landmasked grid cell is NaN (<15% sea ice concentration).
    - Ice particle has stopped moving (|u|<100 m /day for 3 consecutive days).
    - Particle has drifted onto land as defined by *MASK*.
    
- Saving to *output/back_trajectories_m1_v1.p*

*NOTE*: This is not optimized for speed. Takes **days** to run this on a laptop.  
> - Update Jan 2026 - now a little faster (4-5 hours)

In [3]:
#Imports 
import pickle
import datetime
import sys
from cartopy import crs as ccrs, feature as cfeature
import matplotlib.pyplot as plt
from matplotlib.dates import num2date, date2num

#### Load data and class/methods from other notebooks

In [4]:
print('Running other notebooks: Preparing data..\r', end = '')
%run "Load and prepare data.ipynb"

<xarray.Dataset> Size: 3MBreparing data..
Dimensions:          (x: 240, y: 440)
Coordinates:
  * x                (x) float64 2kB -1.058e+05 -9.58e+04 ... 2.284e+06
  * y                (y) float64 4kB -1.706e+06 -1.696e+06 ... 2.684e+06
Data variables:
    lon              (y, x) float32 422kB ...
    lat              (y, x) float32 422kB ...
    landmask         (y, x) float64 845kB ...
    landmask_wo_svb  (y, x) float64 845kB ...
Attributes:
    Conventions:  COARDS, CF-1.5
    title:        Produced by grdconvert
    history:      grdconvert cut/ibcao-v4.2-400m.tiff cut/ibcao-v4.2-400m.nc
    description:  
    GMT_version:  5.4.5 [64-bit]
    node_offset:  1


In [5]:
print('Running other notebooks: Defining particle tracking class..', end = '')
%run "Define particle tracking class.ipynb"
print('  ..done.')

Running other notebooks: Defining particle tracking class..  ..done.


In [6]:
%matplotlib tk

#### Set mooring name and coordinates

In [7]:
moor = 'm2'
lon = 32.319833
lat = 79.678833

#### Set time min and max

In [8]:
time_start =  datetime.datetime(2018, 8, 1, tzinfo=datetime.timezone.utc)
time_end = datetime.datetime(2021, 12, 31, tzinfo=datetime.timezone.utc)
time_gap_days = 2

#### Set output file

In [9]:
datadir_out = './output/'  
savenm =datadir_out + 'back_trajectories_%s_v2.p'%moor
print(savenm)

./output/back_trajectories_m2_v2.p


# Trajectories

#### Define srtuctured input object based on the input dataset


In [10]:
inputs = IceParticleInputs(ds) 

#### Set dictonary with input parameters 
We  will iterate over the variable *time0* - everything else stays the same.

In [11]:
IC_dict0 = {'inputs':inputs, 'lat0':lat, 'lon0':lon, 'time0':None, 'interp_method':'linear', 
              'end_criteria':'ABC', 'MASK':MASK} 

## Evaluation loop

Looping through every *time_gap_days*th day from *time_start* to *time_end* and computing the back-trajectory from M1.

*Used to take days to execute, better now:*

    CPU times: user 4h 21min 57s, sys: 12min 53s, total: 4h 34min 51s
    Wall time: 3h 41min 42s

-> Script allows for picking up an interrupted run *(load_old = True)*


In [14]:
%%time

TRAJS = []
lat0, lon0 =  lat, lon

# Set load_old=True to load previously stored file and pick up where 
# that ends (useful for picking up an interrupted run). 
load_old = True
if load_old:
    with open(savenm, 'rb') as f:
        Ps =pickle.load(f)
    time_ = num2date(date2num(Ps[-1]['time0']))
else:
    time_ = time_start

# - Loop through time steps
# - Storing output (one dictionary per trajectory) in the dictionary TRAJS
# - Saving TRAJS to disk along the way

count = 0

while time_ < time_end:
    
    # STEP FORWARD #
    # Update the input directory
    count += 1
    time_ = time_ + datetime.timedelta(days=float(time_gap_days))
    IC_dict = IC_dict0.copy()
    IC_dict['time0'] = time_
    
    # EVALUATING #
    # Set up object, compute the trajectory, and save to TRAJS
    
    print('### %i / %s.. ### '%(count, time_.strftime('%d %b %Y')))
    P = IceParticle(IC_dict)
    P.backtrack()
    TRAJS += [P.to_dict()]
    
    # SAVING ##
    # Saving all long trajectories (>50 km)
    # Otherwise, save every 10th
    
    save_it = False
    if P.travel_time_days>50:
        save_it = True
    if count//10 == count/10:
        save_it = True
            
    if save_it:
        with open(savenm, 'wb') as f:
            pickle.dump(TRAJS, f)
        print('SAVING!')

        
    sys.stdout.flush()   # forces everything printed so far to appear immediately


### 1 / 23 Jun 2019.. ### 
 ..done.
HIT OPEN WATER! () step 634 (27 Sep 2017) - 1954.3 km ..
SAVING!
### 2 / 25 Jun 2019.. ### 
 ..done.
HIT OPEN WATER! () step 636 (27 Sep 2017) - 1972.2 km ..
SAVING!
### 3 / 27 Jun 2019.. ### 
 ..done.
HIT OPEN WATER! () step 640 (25 Sep 2017) - 1972.2 km ..
SAVING!
### 4 / 29 Jun 2019.. ### 
 ..done.
HIT OPEN WATER! () step 643 (24 Sep 2017) - 1945.0 km ..
SAVING!
### 5 / 01 Jul 2019.. ### 
 ..done.
LAND AHOY! (Particle hit a coastline)017) - 1848.5 km ..
SAVING!
### 6 / 03 Jul 2019.. ### 
 ..done.
LAND AHOY! (Particle hit a coastline)017) - 1811.5 km ..
SAVING!
### 7 / 05 Jul 2019.. ### 
 ..done.
LAND AHOY! (Particle hit a coastline)017) - 1761.7 km ..
SAVING!
### 8 / 07 Jul 2019.. ### 
 ..done.
LAND AHOY! (Particle hit a coastline)017) - 1762.7 km ..
SAVING!
### 9 / 09 Jul 2019.. ### 
 ..done.
LAND AHOY! (Particle hit a coastline)017) - 1756.5 km ..
SAVING!
### 10 / 11 Jul 2019.. ### 
 ..done.
LAND AHOY! (Particle hit a coastline)017) - 1763.2 km 

HIT OPEN WATER! () step 26 (14 Jan 2019) - 53.7 km ..
### 97 / 11 Feb 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 28 (14 Jan 2019) - 49.3 km ..
### 98 / 13 Feb 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 30 (14 Jan 2019) - 45.8 km ..
### 99 / 15 Feb 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 32 (14 Jan 2019) - 44.7 km ..
### 100 / 17 Feb 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 34 (14 Jan 2019) - 11.7 km ..
SAVING!
### 101 / 19 Feb 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 37 (13 Jan 2019) - 14.9 km ..
### 102 / 21 Feb 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 39 (13 Jan 2019) - 13.8 km ..
### 103 / 23 Feb 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 41 (13 Jan 2019) - 13.2 km ..
### 104 / 25 Feb 2019.. ### 
Setting up object ..
 ..done.
NO MOVEMENT! (Particle movement stalled last 3 days).
### 105 / 27 Feb 2019.. ### 
Setting 

HIT OPEN WATER! () step 640 (23 Sep 2017) - 2043.4 km ..
SAVING!
### 165 / 27 Jun 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 636 (29 Sep 2017) - 2084.7 km ..
SAVING!
### 166 / 29 Jun 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 635 (02 Oct 2017) - 2092.7 km ..
SAVING!
### 167 / 01 Jul 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 641 (28 Sep 2017) - 2075.6 km ..
SAVING!
### 168 / 03 Jul 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 641 (30 Sep 2017) - 2071.2 km ..
SAVING!
### 169 / 05 Jul 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 646 (27 Sep 2017) - 2056.1 km ..
SAVING!
### 170 / 07 Jul 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 648 (27 Sep 2017) - 2056.4 km ..
SAVING!
### 171 / 09 Jul 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 650 (27 Sep 2017) - 2056.5 km ..
SAVING!
### 172 / 11 Jul 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN

HIT OPEN WATER! () step 32 (17 Oct 2019) - 73.5 km ..
### 238 / 20 Nov 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 34 (17 Oct 2019) - 75.8 km ..
### 239 / 22 Nov 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 36 (17 Oct 2019) - 74.3 km ..
### 240 / 24 Nov 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 38 (17 Oct 2019) - 73.7 km ..
SAVING!
### 241 / 26 Nov 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 40 (17 Oct 2019) - 78.0 km ..
### 242 / 28 Nov 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 42 (17 Oct 2019) - 89.5 km ..
### 243 / 30 Nov 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 44 (17 Oct 2019) - 105.2 km ..
### 244 / 02 Dec 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 46 (17 Oct 2019) - 120.4 km ..
### 245 / 04 Dec 2019.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 48 (17 Oct 2019) - 123.2 km ..
### 246 / 06 Dec 2019.. ### 
Se

 ..done.
HIT OPEN WATER! () step 549 (01 Oct 2018) - 1589.5 km ..
SAVING!
### 306 / 04 Apr 2020.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 542 (10 Oct 2018) - 1614.7 km ..
SAVING!
### 307 / 06 Apr 2020.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 544 (10 Oct 2018) - 1620.9 km ..
SAVING!
### 308 / 08 Apr 2020.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 546 (10 Oct 2018) - 1653.4 km ..
SAVING!
### 309 / 10 Apr 2020.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 548 (10 Oct 2018) - 1657.8 km ..
SAVING!
### 310 / 12 Apr 2020.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 550 (10 Oct 2018) - 1661.0 km ..
SAVING!
### 311 / 14 Apr 2020.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 552 (10 Oct 2018) - 1647.1 km ..
SAVING!
### 312 / 16 Apr 2020.. ### 
Setting up object ..
 ..done.
HIT OPEN WATER! () step 554 (10 Oct 2018) - 1605.4 km ..
SAVING!
### 313 / 18 Apr 2020.. ### 
Setting up object ..
 ..done.

 ..done.
STARTED IN OPEN WATER!
### 380 / 30 Aug 2020.. ### 
Setting up object ..
 ..done.
STARTED IN OPEN WATER!
SAVING!
### 381 / 01 Sep 2020.. ### 
Setting up object ..
 ..done.
STARTED IN OPEN WATER!
### 382 / 03 Sep 2020.. ### 
Setting up object ..
 ..done.
STARTED IN OPEN WATER!
### 383 / 05 Sep 2020.. ### 
Setting up object ..
 ..done.
STARTED IN OPEN WATER!
### 384 / 07 Sep 2020.. ### 
Setting up object ..
 ..done.
STARTED IN OPEN WATER!
### 385 / 09 Sep 2020.. ### 
Setting up object ..
 ..done.
STARTED IN OPEN WATER!
### 386 / 11 Sep 2020.. ### 
Setting up object ..
 ..done.
STARTED IN OPEN WATER!
### 387 / 13 Sep 2020.. ### 
Setting up object ..
 ..done.
STARTED IN OPEN WATER!
### 388 / 15 Sep 2020.. ### 
Setting up object ..
 ..done.
STARTED IN OPEN WATER!
### 389 / 17 Sep 2020.. ### 
Setting up object ..
 ..done.
STARTED IN OPEN WATER!
### 390 / 19 Sep 2020.. ### 
Setting up object ..
 ..done.
STARTED IN OPEN WATER!
SAVING!
### 391 / 21 Sep 2020.. ### 
Setting up object ..


Can also load if we haev already computed and saved the trajectories.

In [11]:
if False:
    with open(savenm, 'rb') as f:
        TRAJS = pickle.load(f)

### Plot the results

More or less reproductying manuscript figures.

#### Map plot of all trajectories

In [15]:
lat_ctr, lon_ctr = 86, 30

cart_proj = ccrs.Stereographic(
    central_latitude=lat_ctr,
    central_longitude=lon_ctr,
    scale_factor=None, globe=None)

fig = plt.figure(figsize = (8, 8))
ax = plt.axes(projection=cart_proj)

ax.set_facecolor('skyblue'  )
# Coastline
ax.coastlines('50m', color = 'w', lw = 0.5 )

# Land
land_50m = cfeature.NaturalEarthFeature(
    'physical', 'land', '50m',
    edgecolor='k', linewidth = 1,
    facecolor='lightgray')

ax.add_feature(land_50m)

# Set boundaries
ax.set_extent([-20, 191, 66, 89.5], crs=ccrs.PlateCarree())


for P in TRAJS:
    ax.plot(P['lon'], P['lat'], 'k',  transform=ccrs.PlateCarree(), lw = 0.5, alpha = 0.3)


#### Plot of particles by time

##### Classify by end reason


In [16]:
if 'Particle hit open water' in P['end_reason']:
    reason = 1
elif 'movement over 3 days' in P['end_reason']:
    reason = 2
elif '(based on land mask)' in P['end_reason']:
    reason = 3

In [17]:
fig, ax = plt.subplots(2,1, figsize = (10, 3), sharex = True)
for P in TRAJS:
    if P['end_reason']!='Started in open water.':
        
        if 'Particle hit open water' in P['end_reason']:
            col = 'b'
        elif 'movement over 3 days' in P['end_reason']:
            col = 'gray'
        elif '(based on land mask)' in P['end_reason']:
            col = 'r'
        ax[0].plot(num2date(date2num(P['time0'])), P['travel_time_days'], '.', color = col) 
        ax[1].plot(num2date(date2num(P['time0'])), P['total_displacement'], '.', color = col) 

ax[0].grid()
ax[0].grid()
ax[0].text(0.99, 0.86, 'Hit land', ha = 'right', color = 'r', transform=ax[0].transAxes)
ax[0].text(0.99, 0.74, 'Hit open water', ha = 'right', color = 'b', transform=ax[0].transAxes)
ax[0].text(0.99, 0.62, 'Stalled',ha = 'right', color = 'gray', transform=ax[0].transAxes)

Text(0.99, 0.62, 'Stalled')